In [ ]:
import corner
import datetime as dt 
import logging
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import ocnus_py
from ocnus_py import ECHModel, ECHMagFilter, Obs, Univariate, ObsVecNoise

logging.basicConfig(format='%(levelname)s %(filename)s:%(lineno)d %(asctime)-15s %(message)s')
logging.getLogger().setLevel(logging.INFO)

In [2]:
# Set up model priors
model = ECHModel([
    Univariate.uniform("phi", -1.0, 1.0),
    Univariate.uniform("theta", np.pi - 1.25, np.pi + 1.25),
    Univariate.constant("psi", 0.0),
    Univariate.uniform("y_0", -0.5, 0.5),
    Univariate.constant("cs_delta", 1.0),
    Univariate.constant("radius", 0.11),
    Univariate.uniform("x_0", 0.25, 1.0),
    Univariate.constant("speed", 500.0),
    Univariate.uniform("b_scale", 10, 25),
    Univariate.constant("lambda", 0.0),
    Univariate.constant("alpha", 0.0),
    Univariate.uniform("tau", 0.0, 20.0),
])

ref_data = np.array([
    [np.nan, np.nan, np.nan],
    [-3.13531505, -13.98423491, -6.64658269],
    [-3.38360946, -15.34720671, -1.71139834],
    [-4.21540068, -13.93568105, 4.73878965],
    [-5.41469694, -13.9772912, 4.14112123],
    [-4.30711573, -12.61217154, 5.78382821],
    [np.nan, np.nan, np.nan],
])

# set up observation
obs = Obs(
    [4.0 * 3600.0 * i for i in range(len(ref_data))],
    np.array([[1.0, 0.0, 0.0] for _ in range(len(ref_data))]),
)

filter = model.new_mag_filter(obs, ref_data, 2048, 42)
filter.initialize(metric='nrmse', threshold=1.0)

particles_init = filter.particles()

DEBUG base.rs:163 2026-02-11 11:00:13,090 initialize_ensbl: 16.4k evaluations in 4ms
DEBUG base.rs:420 2026-02-11 11:00:13,134 simulate_ensbl: 114.7k evaluations in 27ms
DEBUG base.rs:163 2026-02-11 11:00:13,228 initialize_ensbl: 16.4k evaluations in 50ms
DEBUG base.rs:420 2026-02-11 11:00:13,261 simulate_ensbl: 114.7k evaluations in 32ms
DEBUG base.rs:163 2026-02-11 11:00:13,301 initialize_ensbl: 16.4k evaluations in 10ms
DEBUG base.rs:420 2026-02-11 11:00:13,331 simulate_ensbl: 114.7k evaluations in 28ms
DEBUG base.rs:163 2026-02-11 11:00:13,370 initialize_ensbl: 16.4k evaluations in 1ms
DEBUG base.rs:420 2026-02-11 11:00:13,405 simulate_ensbl: 114.7k evaluations in 34ms
DEBUG base.rs:163 2026-02-11 11:00:13,479 initialize_ensbl: 16.4k evaluations in 3ms
DEBUG base.rs:420 2026-02-11 11:00:13,528 simulate_ensbl: 114.7k evaluations in 35ms
DEBUG base.rs:163 2026-02-11 11:00:13,544 initialize_ensbl: 16.4k evaluations in 2ms
DEBUG base.rs:420 2026-02-11 11:00:13,600 simulate_ensbl: 114.7

In [ ]:
filter_abc = filter.copy()

noise = ObsVecNoise.additive_normal(std_dev=1.0)

for threshold in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.175, 0.15]:
    ess = filter_abc.abc_mvnk(metric='nrmse', threshold=threshold, noise=noise)
    print("Filtering with threshold {}, effective sample size: {:.1f}".format(threshold, ess))

print("ABC Mean: {:.3f}".format(np.mean(filter_abc.errors())))
print("ABC Minimum: {:.3f}".format(np.min(filter_abc.errors())))

particles_abc = filter_abc.particles()

DEBUG base.rs:163 2026-02-11 11:00:15,863 initialize_ensbl: 16.4k evaluations in 51ms
DEBUG base.rs:420 2026-02-11 11:00:15,898 simulate_ensbl: 114.7k evaluations in 34ms
DEBUG mod.rs:226 2026-02-11 11:00:15,904 removing excessive ensemble members(n=1305)
DEBUG abc.rs:143 2026-02-11 11:00:16,016 abc_iter
	eps: 0.585 -- 0.668 -- 0.748
	ran 0.115M evaluations in 0.20 sec
	effective sample size = 1274.6 / 2048
DEBUG base.rs:277 2026-02-11 11:00:16,017 initialize_states_ensbl: 2.0k evaluations in 0ms
DEBUG base.rs:420 2026-02-11 11:00:16,021 simulate_ensbl: 14.3k evaluations in 3ms
DEBUG base.rs:163 2026-02-11 11:00:16,036 initialize_ensbl: 16.4k evaluations in 12ms
DEBUG base.rs:420 2026-02-11 11:00:16,073 simulate_ensbl: 114.7k evaluations in 36ms
DEBUG mod.rs:226 2026-02-11 11:00:16,076 removing excessive ensemble members(n=836)


Filtering with threshold 0.9, effective sample size: 1274.6


DEBUG abc.rs:143 2026-02-11 11:00:16,276 abc_iter
	eps: 0.536 -- 0.611 -- 0.681
	ran 0.115M evaluations in 0.25 sec
	effective sample size = 1532.9 / 2048
DEBUG base.rs:277 2026-02-11 11:00:16,296 initialize_states_ensbl: 2.0k evaluations in 8ms
DEBUG base.rs:420 2026-02-11 11:00:16,339 simulate_ensbl: 14.3k evaluations in 3ms
DEBUG base.rs:163 2026-02-11 11:00:16,368 initialize_ensbl: 16.4k evaluations in 18ms
DEBUG base.rs:420 2026-02-11 11:00:16,399 simulate_ensbl: 114.7k evaluations in 29ms
DEBUG mod.rs:226 2026-02-11 11:00:16,403 removing excessive ensemble members(n=602)


Filtering with threshold 0.8, effective sample size: 1532.9


DEBUG abc.rs:143 2026-02-11 11:00:16,584 abc_iter
	eps: 0.475 -- 0.536 -- 0.598
	ran 0.115M evaluations in 0.23 sec
	effective sample size = 1647.0 / 2048
DEBUG base.rs:277 2026-02-11 11:00:16,585 initialize_states_ensbl: 2.0k evaluations in 0ms
DEBUG base.rs:420 2026-02-11 11:00:16,592 simulate_ensbl: 14.3k evaluations in 6ms
DEBUG base.rs:163 2026-02-11 11:00:16,629 initialize_ensbl: 16.4k evaluations in 33ms
DEBUG base.rs:420 2026-02-11 11:00:16,660 simulate_ensbl: 114.7k evaluations in 30ms
DEBUG mod.rs:226 2026-02-11 11:00:16,665 removing excessive ensemble members(n=330)
DEBUG abc.rs:143 2026-02-11 11:00:16,765 abc_iter
	eps: 0.418 -- 0.471 -- 0.521
	ran 0.115M evaluations in 0.17 sec
	effective sample size = 1748.7 / 2048
DEBUG base.rs:277 2026-02-11 11:00:16,767 initialize_states_ensbl: 2.0k evaluations in 0ms
DEBUG base.rs:420 2026-02-11 11:00:16,773 simulate_ensbl: 14.3k evaluations in 6ms


...

Filtering with threshold 0.2, effective sample size: 1779.9


DEBUG base.rs:420 2026-02-11 11:00:19,057 simulate_ensbl: 114.7k evaluations in 32ms
DEBUG base.rs:163 2026-02-11 11:00:19,076 initialize_ensbl: 16.4k evaluations in 8ms
DEBUG base.rs:420 2026-02-11 11:00:19,111 simulate_ensbl: 114.7k evaluations in 34ms
DEBUG base.rs:163 2026-02-11 11:00:19,126 initialize_ensbl: 16.4k evaluations in 11ms
DEBUG base.rs:420 2026-02-11 11:00:19,155 simulate_ensbl: 114.7k evaluations in 29ms
DEBUG base.rs:163 2026-02-11 11:00:19,166 initialize_ensbl: 16.4k evaluations in 8ms
DEBUG base.rs:420 2026-02-11 11:00:19,195 simulate_ensbl: 114.7k evaluations in 28ms
DEBUG base.rs:163 2026-02-11 11:00:19,207 initialize_ensbl: 16.4k evaluations in 9ms
DEBUG base.rs:420 2026-02-11 11:00:19,246 simulate_ensbl: 114.7k evaluations in 38ms
DEBUG base.rs:163 2026-02-11 11:00:19,262 initialize_ensbl: 16.4k evaluations in 13ms
DEBUG base.rs:420 2026-02-11 11:00:19,301 simulate_ensbl: 114.7k evaluations in 38ms
DEBUG base.rs:163 2026-02-11 11:00:19,315 initialize_ensbl: 16.

Filtering with threshold 0.175, effective sample size: 1640.5


DEBUG base.rs:163 2026-02-11 11:00:19,709 initialize_ensbl: 16.4k evaluations in 8ms
DEBUG base.rs:420 2026-02-11 11:00:19,737 simulate_ensbl: 114.7k evaluations in 27ms
DEBUG base.rs:163 2026-02-11 11:00:19,798 initialize_ensbl: 16.4k evaluations in 32ms
DEBUG base.rs:420 2026-02-11 11:00:19,834 simulate_ensbl: 114.7k evaluations in 34ms
DEBUG base.rs:163 2026-02-11 11:00:19,893 initialize_ensbl: 16.4k evaluations in 20ms
DEBUG base.rs:420 2026-02-11 11:00:19,933 simulate_ensbl: 114.7k evaluations in 39ms
DEBUG base.rs:163 2026-02-11 11:00:19,972 initialize_ensbl: 16.4k evaluations in 32ms
DEBUG base.rs:420 2026-02-11 11:00:20,007 simulate_ensbl: 114.7k evaluations in 34ms
DEBUG base.rs:163 2026-02-11 11:00:20,082 initialize_ensbl: 16.4k evaluations in 17ms
DEBUG base.rs:420 2026-02-11 11:00:20,118 simulate_ensbl: 114.7k evaluations in 35ms
DEBUG base.rs:163 2026-02-11 11:00:20,144 initialize_ensbl: 16.4k evaluations in 9ms
DEBUG base.rs:420 2026-02-11 11:00:20,181 simulate_ensbl: 114

Filtering with threshold 0.15, effective sample size: 1349.4
ABC Mean: 0.161
ABC Minimum: 0.133


In [4]:
filter_dev = filter.copy()

mutation_counter = 0

for i in range(800):
    mutation_counter += filter_dev.dev(metric='nchisq', mutation_factor=0.1, recombination_factor=0.8)

    if i > 0 and i % 100 == 0:
        print("Last 100 iterations: {} mutations".format(mutation_counter))
        mutation_counter = 0

print("DEV Minimum: {:.3f}".format(np.min(filter_dev.errors())))

particles_dev = filter_dev.particles()

dev_min = particles_dev[0][np.argmin(filter_dev.errors())]

Last 100 iterations: 45117 mutations
Last 100 iterations: 32542 mutations
Last 100 iterations: 17955 mutations
Last 100 iterations: 13445 mutations
Last 100 iterations: 9830 mutations
Last 100 iterations: 15692 mutations
Last 100 iterations: 9707 mutations
DEV Minimum: 0.017


In [5]:
autorange = np.array([np.min(particles_init[0], axis=0), np.max(particles_init[0], axis=0)]).T
autoflag = [True for _ in autorange]

for idx in range(len(autorange)):
        if autorange[idx][0] == autorange[idx][1]:
                autoflag[idx] = False

cfig = plt.figure(figsize=(6, 6))
corner.corner(particles_init[0].T[autoflag].T, 
              weights=np.ones_like(particles_abc[1]) / 2048, color='k',
              range=autorange[autoflag], fig=cfig, smooth=5.0, 
              smooth1d=1.0, bins=20, hist_bin_factor=2)
corner.corner(particles_abc[0].T[autoflag].T, 
              weights=particles_abc[1], color='r',
              range=autorange[autoflag], fig=cfig, smooth=2.0, 
              smooth1d=1.0, bins=20, hist_bin_factor=2, show_titles=True)
corner.corner(particles_dev[0].T[autoflag].T, 
              weights=np.ones_like(particles_abc[1]) / 2048, color='g',
              range=autorange[autoflag], fig=cfig, smooth=2.0, 
              smooth1d=1.0, bins=20, hist_bin_factor=2, show_titles=True)
cfig.legend(handles=[
    Patch(facecolor='k', edgecolor='k', label="Initial"),
    Patch(facecolor='r', edgecolor='k', label="Approximate Bayesian Computation (ABC)"),
    Patch(facecolor='g', edgecolor='k', label="Differential Evolution (DEV)"),
], loc='upper right')
plt.show()

In [6]:
filter_sir = filter_abc.copy()

for i in [5, 4, 3, 2, 1.5, 1, 1, 1]:
    (ess, us) = filter_sir.sir_mvnk(i * np.eye(7))
    print("SIR Filtering with covariance scale {}, effective sample size: {:.1f}, unique samples {}".format(i, ess, us))

particles_sir = filter_sir.particles()

SIR Filtering with covariance scale 5, effective sample size: 3395.9, unique samples 1592
SIR Filtering with covariance scale 4, effective sample size: 2396.5, unique samples 1474
SIR Filtering with covariance scale 3, effective sample size: 2214.9, unique samples 1448
SIR Filtering with covariance scale 2, effective sample size: 1552.8, unique samples 1272
SIR Filtering with covariance scale 1.5, effective sample size: 1961.0, unique samples 1385
SIR Filtering with covariance scale 1, effective sample size: 1236.7, unique samples 1200
SIR Filtering with covariance scale 1, effective sample size: 3249.7, unique samples 1586
SIR Filtering with covariance scale 1, effective sample size: 2876.6, unique samples 1575


In [7]:
autorange = np.array([np.min(particles_abc[0], axis=0), np.max(particles_abc[0], axis=0)]).T
autoflag = [True for _ in autorange]

for idx in range(len(autorange)):
        if autorange[idx][0] == autorange[idx][1]:
                autoflag[idx] = False

cfig = plt.figure(figsize=(6, 6))
corner.corner(particles_abc[0].T[autoflag].T, 
              weights=particles_abc[1], color='r',
              range=autorange[autoflag], fig=cfig, smooth=5.0, 
              smooth1d=1.0, bins=20, hist_bin_factor=2)
corner.corner(particles_sir[0].T[autoflag].T, 
              weights=np.ones_like(particles_abc[1]) / 2048, color='g',
              range=autorange[autoflag], fig=cfig, smooth=2.0, 
              smooth1d=1.0, bins=20, hist_bin_factor=2, show_titles=True)
cfig.legend(handles=[
    Patch(facecolor='k', edgecolor='k', label="Initial"),
    Patch(facecolor='r', edgecolor='k', label="Approximate Bayesian Computation (ABC)"),
    Patch(facecolor='g', edgecolor='k', label="Sequential Importance Resampling (SIR)"),
], loc='upper right')
plt.show()